In [ ]:
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition
import pyomo.environ as pyo
import pandas as pd

# ── Dados ──────────────────────────────────────────────────────────────────────
df = pd.DataFrame({
    'tempo': pd.to_datetime(list(range(0, 24)), unit='h',
                            origin='2026-03-13').strftime('%H:%M').tolist(),
    'P Demand': [
        1.9317, 1.6090, 1.4079, 1.3281, 1.3834, 1.6413,
        1.9395, 1.7383, 1.8341, 1.8354, 1.9312, 2.3645,
        2.2038, 2.2997, 2.1659, 2.5046, 2.7490, 4.0597,
        4.9924, 5.4257, 5.0491, 4.4294, 3.7692, 2.7716
    ],
    "P PV": [
        0.0000, 0.0000, 0.0000, 0.0000, 0.0796, 0.4565,
        1.0742, 1.5790, 2.4343, 2.7488, 3.5092, 3.8988,
        3.9734, 3.7105, 3.1671, 2.7282, 2.3926, 2.1764,
        1.9083, 1.4257, 0.0034, 0.0000, 0.0000, 0.0000
    ],
    "tariff": [
        0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
        0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
        0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.32629,
        0.51792, 0.51792, 0.51792, 0.32629, 0.22419, 0.22419
    ]
})

parameters = {
    "General": {"timestep": 60},
    "Grid":    {"Pmax": 100},
    "BESS": {
        "Pmax": 2.0,
        "eff": 0.90,
        "self_discharge": 0.05,
        "capacity": 7,
        "initial capacity": 0,
    }
}

df.set_index('tempo', inplace=True)


# ── Classes ────────────────────────────────────────────────────────────────────
class General:
    def __init__(self, parameters):
        self.timestep = parameters["General"]["timestep"]

class BESS:
    def __init__(self, parameters, general):
        self.general          = general
        self.Pmax             = parameters["BESS"]["Pmax"]
        self.eff              = parameters["BESS"]["eff"]
        self.initial_capacity = parameters["BESS"]["initial capacity"]
        self.capacity         = parameters["BESS"]["capacity"]
        self.self_discharge   = parameters["BESS"]["self_discharge"]

class Grid:
    def __init__(self, parameters, general):
        self.general = general
        self.Pmax    = parameters["Grid"]["Pmax"]

class PV:
    def __init__(self, parameters, general):
        self.general = general

class SmartHome:
    def __init__(self, parameters, df):
        self.df      = df
        self.general = General(parameters)
        self.grid    = Grid(parameters, self.general)
        self.bess    = BESS(parameters, self.general)
        self.pv      = PV(parameters, self.general)

    def build(self):
        model = pyo.ConcreteModel('SmartHome')
        T     = self.df.index.tolist()   # ['00:00', '01:00', ..., '23:00']
        delta = self.general.timestep / 60  # 1.0 hora

        # ── Variáveis ──────────────────────────────────────────────────────────
        # Potência comprada (+) ou vendida (-) da rede
        model.Pgrid = pyo.Var(T,
                              bounds=(0, self.grid.Pmax),
                              within=pyo.Reals)

        # Potência vendida à rede (injeção de excedente PV)
        model.Pgrid_sell = pyo.Var(T,
                                   bounds=(0, self.grid.Pmax),
                                   within=pyo.Reals)

        # Potência de carga da bateria (carregando)
        model.Pbess_charge = pyo.Var(T,
                                     bounds=(0, self.bess.Pmax),
                                     within=pyo.Reals)

        # Potência de descarga da bateria (descarregando)
        model.Pbess_discharge = pyo.Var(T,
                                        bounds=(0, self.bess.Pmax),
                                        within=pyo.Reals)

        # Energia armazenada na bateria
        model.E_bess = pyo.Var(T,
                               bounds=(0, self.bess.capacity),
                               within=pyo.Reals)

        # Estado da bateria: 1 = descarregando, 0 = carregando
        model.state = pyo.Var(T, within=pyo.Binary)

        # ── Restrições ─────────────────────────────────────────────────────────

        # 1) Balanço de potência em cada hora
        #    Pgrid + PV - Pgrid_sell = Demanda + Pbess_charge - Pbess_discharge
        def power_balance_rule(model, t):
            return (  model.Pgrid[t]
                    + self.df.loc[t, 'P PV']
                    - model.Pgrid_sell[t]
                    ==
                      self.df.loc[t, 'P Demand']
                    + model.Pbess_charge[t]
                    - model.Pbess_discharge[t])

        model.power_balance = pyo.Constraint(T, rule=power_balance_rule)

        # 2) Dinâmica da bateria hora a hora
        #    E[t] = E[t-1] + eff*delta*Pch[t] - delta*Pdis[t]/eff - beta*delta*E[t]
        def bess_energy_rule(model, t):
            idx  = T.index(t)
            eff  = self.bess.eff
            beta = self.bess.self_discharge

            charge_term    = eff  * delta * model.Pbess_charge[t]
            discharge_term = delta * model.Pbess_discharge[t] / eff
            loss_term      = beta * delta * model.E_bess[t]

            if idx == 0:
                E_prev = self.bess.initial_capacity  # hora 0: sem hora anterior
            else:
                E_prev = model.E_bess[T[idx - 1]]   # hora anterior

            return model.E_bess[t] == E_prev + charge_term - discharge_term - loss_term

        model.bess_energy = pyo.Constraint(T, rule=bess_energy_rule)

        # 3) Bateria só pode carregar OU descarregar (não os dois ao mesmo tempo)
        #    Pbess_discharge <= Pmax *  state[t]       (só descarrega se state=1)
        #    Pbess_charge    <= Pmax * (1 - state[t])  (só carrega   se state=0)
        def bess_discharge_limit(model, t):
            return model.Pbess_discharge[t] <= self.bess.Pmax * model.state[t]

        def bess_charge_limit(model, t):
            return model.Pbess_charge[t] <= self.bess.Pmax * (1 - model.state[t])

        model.bess_dis_limit = pyo.Constraint(T, rule=bess_discharge_limit)
        model.bess_ch_limit  = pyo.Constraint(T, rule=bess_charge_limit)

        # ── Função objetivo ────────────────────────────────────────────────────
        # Minimizar custo total de compra de energia da rede
        def objective_rule(model):
            return delta * sum(self.df.loc[t, 'tariff'] * model.Pgrid[t]
                               for t in T)

        model.funcao_objetivo = pyo.Objective(rule=objective_rule,
                                              sense=pyo.minimize)
        self.model = model

    def solve(self):
        solver   = SolverFactory('glpk')
        solution = solver.solve(self.model, tee=True)  # tee=True imprime o log

        # Verifica se resolveu com sucesso
        if (solution.solver.status == SolverStatus.ok and
            solution.solver.termination_condition == TerminationCondition.optimal):
            print("\n✓ Solução ótima encontrada!")
            print(f"  Custo total: R$ {pyo.value(self.model.funcao_objetivo):.4f}")
        else:
            print("✗ Solver não encontrou solução ótima.")

        return solution


# ── Execução ───────────────────────────────────────────────────────────────────
smarthome = SmartHome(parameters, df)
smarthome.build()
smarthome.solve()